<a href="https://colab.research.google.com/github/MariaAkterKhadiza/deep_learning_7th-semeter/blob/main/thesis_potato_leaf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q torch torchvision torchaudio
!pip install -q timm
!pip install -q albumentations
!pip install -q opencv-python
!pip install -q scikit-learn
!pip install -q matplotlib
!pip install -q seaborn
!pip install -q grad-cam
!pip install -q ultralytics
!pip install -q streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 48.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 113.4 MB/s eta 0:00:00


In [2]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader

import timm

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [3]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
dataset_path="/content/drive/MyDrive/potato_leaf(dataset)/Potato_Leaf_Disease-20260701T072454Z-3-001"

In [ ]:


import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image

from sklearn.model_selection import train_test_split

from PIL import Image

print("TensorFlow Version :",tf.__version__)

In [ ]:
print("GPU Available :",tf.config.list_physical_devices('GPU'))

In [ ]:
dataset_path="/content/drive/MyDrive/potato_leaf(dataset)/Potato_Leaf_Disease-20260701T072454Z-3-001/Potato_Leaf_Disease"

In [ ]:
os.listdir(dataset_path)

In [ ]:
classes=sorted(os.listdir(dataset_path))

print(classes)

In [ ]:
print("Number of Classes :",len(classes))

In [ ]:
print("Dataset Information")
print("="*40)

total_images=0

for cls in classes:

    folder=os.path.join(dataset_path,cls)

    count=len(os.listdir(folder))

    total_images+=count

    print(f"{cls:<30}{count}")

print("="*40)

print("Total Images :",total_images)

In [ ]:
plt.figure(figsize=(20,10))

for i,cls in enumerate(classes):

    folder=os.path.join(dataset_path,cls)

    img_name=random.choice(os.listdir(folder))

    img_path=os.path.join(folder,img_name)

    img=image.load_img(img_path,target_size=(224,224))

    plt.subplot(2,4,i+1)

    plt.imshow(img)

    plt.title(cls.replace("Potato___",""))

    plt.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
sample=os.path.join(dataset_path,
                    classes[0],
                    os.listdir(os.path.join(dataset_path,classes[0]))[0])

img=Image.open(sample)

print("Original Image Size :",img.size)

In [ ]:
print("Image Mode :",img.mode)

print("Image Format :",img.format)

In [ ]:
image_paths=[]
labels=[]

for cls in classes:

    folder=os.path.join(dataset_path,cls)

    for img in os.listdir(folder):

        image_paths.append(os.path.join(folder,img))

        labels.append(cls)

In [ ]:
df=pd.DataFrame({
    "filepath":image_paths,
    "label":labels
})

df.head()

In [ ]:
print(df.shape)

In [ ]:
df["label"].value_counts()

In [ ]:
plt.figure(figsize=(6,4))

df["label"].value_counts().plot(kind="bar")

plt.title("Class Distribution")

plt.xlabel("Disease")

plt.ylabel("Images")

plt.show()

In [ ]:
rescale=1./255
target_size=(224,224)

In [ ]:
train_datagen=ImageDataGenerator(

    rescale=1./255,

    rotation_range=20,

    width_shift_range=0.2,

    height_shift_range=0.2,

    zoom_range=0.2,

    shear_range=0.2,

    horizontal_flip=True,

    fill_mode="nearest"

)

In [ ]:
test_datagen=ImageDataGenerator(

    rescale=1./255

)

In [ ]:
train_df,temp_df=train_test_split(

    df,

    test_size=0.30,

    random_state=42,

    stratify=df["label"]

)

In [ ]:
valid_df,test_df=train_test_split(

    temp_df,

    test_size=0.50,

    random_state=42,

    stratify=temp_df["label"]

)

In [ ]:
print("Training :",train_df.shape)

print("Validation :",valid_df.shape)

print("Testing :",test_df.shape)

In [ ]:
batch_size=32

image_size=(224,224)

train_generator=train_datagen.flow_from_dataframe(

    train_df,

    x_col="filepath",

    y_col="label",

    target_size=image_size,

    batch_size=batch_size,

    class_mode="categorical",

    shuffle=True

)

In [ ]:
validation_generator=test_datagen.flow_from_dataframe(

    valid_df,

    x_col="filepath",

    y_col="label",

    target_size=image_size,

    batch_size=batch_size,

    class_mode="categorical",

    shuffle=False

)

In [ ]:
test_generator=test_datagen.flow_from_dataframe(

    test_df,

    x_col="filepath",

    y_col="label",

    target_size=image_size,

    batch_size=batch_size,

    class_mode="categorical",

    shuffle=False

)

In [ ]:
print(train_generator.class_indices)

In [ ]:
images,labels=next(train_generator)

plt.figure(figsize=(15,8))

for i in range(9):

    plt.subplot(3,3,i+1)

    plt.imshow(images[i])

    plt.title(classes[np.argmax(labels[i])].replace("Potato___",""))

    plt.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
print("="*40)

print("Training Images :",train_generator.samples)

print("Validation Images :",validation_generator.samples)

print("Testing Images :",test_generator.samples)

print("="*40)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.models import Model

from tensorflow.keras.layers import *

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import ModelCheckpoint

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications import EfficientNetB0

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    min_lr=1e-6
)

In [ ]:
NUM_CLASSES = len(classes)

print(NUM_CLASSES)

**CNN**

In [ ]:
cnn_model = Sequential([

    Conv2D(32,(3,3),activation="relu",input_shape=(224,224,3)),
    MaxPooling2D(2,2),

    Conv2D(64,(3,3),activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(128,(3,3),activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(256,(3,3),activation="relu"),
    MaxPooling2D(2,2),

    Flatten(),

    Dense(512,activation="relu"),

    Dropout(0.5),

    Dense(NUM_CLASSES,activation="softmax")

])

In [ ]:
cnn_model.compile(

    optimizer=Adam(learning_rate=0.0001),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [ ]:
cnn_model.summary()

In [ ]:
cnn_checkpoint = ModelCheckpoint(

    "CNN_Best.keras",

    monitor="val_accuracy",

    save_best_only=True,

    mode="max"

)

**TRAIN_CNN**

In [ ]:
cnn_history = cnn_model.fit(

    train_generator,

    validation_data=validation_generator,

    epochs=5,

    callbacks=[

        early_stop,

        reduce_lr,

        cnn_checkpoint

    ]

)

**ResNet50**

In [ ]:
base_model = ResNet50(

    include_top=False,

    weights="imagenet",

    input_shape=(224,224,3)

)

**Freeze Layer**

In [ ]:
for layer in base_model.layers:

    layer.trainable=False

In [ ]:
x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dense(256,activation="relu")(x)

x = Dropout(0.5)(x)

predictions = Dense(NUM_CLASSES,activation="softmax")(x)

resnet_model = Model(

    inputs=base_model.input,

    outputs=predictions

)

In [ ]:
resnet_model.compile(

    optimizer=Adam(1e-4),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

resnet_checkpoint = ModelCheckpoint(

    "ResNet50_Best.keras",

    monitor="val_accuracy",

    save_best_only=True

)

In [ ]:
resnet_history = resnet_model.fit(

    train_generator,

    validation_data=validation_generator,

    epochs=20,

    callbacks=[

        early_stop,

        reduce_lr,

        resnet_checkpoint

    ]

)

**MobileNetV2**

In [ ]:
base_model = MobileNetV2(

    include_top=False,

    weights="imagenet",

    input_shape=(224,224,3)

)
for layer in base_model.layers:

    layer.trainable=False

In [ ]:
x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dense(256,activation="relu")(x)

x = Dropout(0.5)(x)

outputs = Dense(NUM_CLASSES,activation="softmax")(x)

mobilenet_model = Model(

    inputs=base_model.input,

    outputs=outputs

)

In [ ]:
mobilenet_model.compile(

    optimizer=Adam(1e-4),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [ ]:
mobilenet_checkpoint = ModelCheckpoint(

    "MobileNetV2_Best.keras",

    save_best_only=True,

    monitor="val_accuracy"

)

In [ ]:
mobilenet_history = mobilenet_model.fit(

    train_generator,

    validation_data=validation_generator,

    epochs=20,

    callbacks=[

        early_stop,

        reduce_lr,

        mobilenet_checkpoint

    ]

)

**EfficientNetB0**

In [ ]:
base_model = EfficientNetB0(

    include_top=False,

    weights="imagenet",

    input_shape=(224,224,3)

)
for layer in base_model.layers:

    layer.trainable=False

In [ ]:
x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dense(256,activation="relu")(x)

x = Dropout(0.5)(x)

outputs = Dense(NUM_CLASSES,activation="softmax")(x)

efficientnet_model = Model(

    inputs=base_model.input,

    outputs=outputs

)

In [ ]:
efficientnet_model.compile(

    optimizer=Adam(1e-4),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [ ]:
efficient_checkpoint = ModelCheckpoint(

    "EfficientNetB0_Best.keras",

    monitor="val_accuracy",

    save_best_only=True

)

In [ ]:
efficient_history = efficientnet_model.fit(

    train_generator,

    validation_data=validation_generator,

    epochs=20,

    callbacks=[

        early_stop,

        reduce_lr,

        efficient_checkpoint

    ]

)

In [ ]:
histories = {
    "CNN": cnn_history,
    "ResNet50": resnet_history,
    "MobileNetV2": mobilenet_history,
    "EfficientNetB0": efficient_history
}

Compare_4 **model**

In [ ]:
from tensorflow.keras.models import load_model

cnn_model = load_model("CNN_Best.keras")

resnet_model = load_model("ResNet50_Best.keras")

mobilenet_model = load_model("MobileNetV2_Best.keras")

efficientnet_model = load_model("EfficientNetB0_Best.keras")

print("All models loaded successfully.")

In [ ]:
models = {

    "CNN": cnn_model,

    "ResNet50": resnet_model,

    "MobileNetV2": mobilenet_model,

    "EfficientNetB0": efficientnet_model

}

In [ ]:
results = {}

for name, model in models.items():

    loss, accuracy = model.evaluate(
        test_generator,
        verbose=1
    )

    results[name] = {

        "Loss": loss,

        "Accuracy": accuracy

    }

print(results)

In [ ]:
import pandas as pd

comparison_df = pd.DataFrame(results).T

comparison_df

In [ ]:
comparison_df = comparison_df.sort_values(
    by="Accuracy",
    ascending=False
)

comparison_df

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.bar(
    comparison_df.index,
    comparison_df["Accuracy"]
)

plt.title("Model Accuracy Comparison")

plt.ylabel("Accuracy")

plt.xlabel("Models")

plt.grid(axis="y")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(
    comparison_df.index,
    comparison_df["Loss"]
)

plt.title("Model Loss Comparison")

plt.ylabel("Loss")

plt.xlabel("Models")

plt.grid(axis="y")

plt.show()

In [ ]:
best_model_name = comparison_df.index[0]

print("Best Model :", best_model_name)

In [ ]:
best_accuracy = comparison_df.iloc[0]["Accuracy"]

print(f"Best Accuracy : {best_accuracy*100:.2f}%")

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report

In [ ]:
y_true = test_generator.classes

print(y_true[:10])

In [ ]:
class_names = list(test_generator.class_indices.keys())

print(class_names)

In [ ]:
def evaluate_model(model, model_name):

    # Predict probabilities
    y_pred_prob = model.predict(test_generator, verbose=1)

    # Convert probabilities to predicted class
    y_pred = y_pred_prob.argmax(axis=1)

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        average="weighted"
    )

    recall = recall_score(
        y_true,
        y_pred,
        average="weighted"
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average="weighted"
    )

    print("="*60)
    print(model_name)
    print("="*60)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")

    print("\nClassification Report\n")

    print(
        classification_report(
            y_true,
            y_pred,
            target_names=class_names
        )
    )

    return accuracy, precision, recall, f1

EVALUATE **CNN**

In [ ]:
cnn_acc, cnn_pre, cnn_rec, cnn_f1 = evaluate_model(

    cnn_model,

    "CNN"

)

Evaluate **ResNet50**

In [ ]:
resnet_acc, resnet_pre, resnet_rec, resnet_f1 = evaluate_model(

    resnet_model,

    "ResNet50"

)

Evaluate **MobileNetV2**

In [ ]:
mobile_acc, mobile_pre, mobile_rec, mobile_f1 = evaluate_model(

    mobilenet_model,

    "MobileNetV2"

)

Evaluate **EfficientNetB0**

In [ ]:
efficient_acc, efficient_pre, efficient_rec, efficient_f1 = evaluate_model(

    efficientnet_model,

    "EfficientNetB0"

)

Create Evaluation **Table**

In [ ]:
evaluation_df = pd.DataFrame({

    "Model":[

        "CNN",

        "ResNet50",

        "MobileNetV2",

        "EfficientNetB0"

    ],

    "Accuracy":[

        cnn_acc,

        resnet_acc,

        mobile_acc,

        efficient_acc

    ],

    "Precision":[

        cnn_pre,

        resnet_pre,

        mobile_pre,

        efficient_pre

    ],

    "Recall":[

        cnn_rec,

        resnet_rec,

        mobile_rec,

        efficient_rec

    ],

    "F1 Score":[

        cnn_f1,

        resnet_f1,

        mobile_f1,

        efficient_f1

    ]

})

evaluation_df

In [ ]:
evaluation_df = evaluation_df.sort_values(

    by="Accuracy",

    ascending=False

)

evaluation_df

In [ ]:
evaluation_df.to_csv(

    "Model_Comparison.csv",

    index=False

)

print("Evaluation results saved successfully.")

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(

    evaluation_df["Model"],

    evaluation_df["Accuracy"]

)

plt.title("Accuracy Comparison")

plt.ylabel("Accuracy")

plt.grid(axis="y")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(

    evaluation_df["Model"],

    evaluation_df["F1 Score"]

)

plt.title("F1 Score Comparison")

plt.ylabel("F1 Score")

plt.grid(axis="y")

plt.show()

In [ ]:
best_model = evaluation_df.iloc[0]

print(best_model)

**Confusion** Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

updated evaluation **function**

In [ ]:
def evaluate_model(model, model_name):

    # Reset generator
    test_generator.reset()

    # Predict
    y_pred_prob = model.predict(
        test_generator,
        verbose=1
    )

    # Predicted Class
    y_pred = np.argmax(
        y_pred_prob,
        axis=1
    )

    # Metrics
    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        average="weighted"
    )

    recall = recall_score(
        y_true,
        y_pred,
        average="weighted"
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average="weighted"
    )

    print("="*60)
    print(model_name)
    print("="*60)

    print("Accuracy :",accuracy)
    print("Precision:",precision)
    print("Recall   :",recall)
    print("F1 Score :",f1)

    print(classification_report(
        y_true,
        y_pred,
        target_names=class_names
    ))

    return (
        accuracy,
        precision,
        recall,
        f1,
        y_pred,
        y_pred_prob
    )

**CNN**

In [ ]:
(
cnn_acc,
cnn_pre,
cnn_rec,
cnn_f1,
cnn_pred,
cnn_prob
)=evaluate_model(

cnn_model,

"CNN"

)

# resnet50

In [ ]:
(
resnet_acc,
resnet_pre,
resnet_rec,
resnet_f1,
resnet_pred,
resnet_prob
)=evaluate_model(

resnet_model,

"ResNet50"

)

**MobileNetV2**

In [ ]:
(
mobile_acc,
mobile_pre,
mobile_rec,
mobile_f1,
mobile_pred,
mobile_prob
)=evaluate_model(

mobilenet_model,

"MobileNetV2"

)

**EFFicientNETB0**

In [ ]:
(
efficient_acc,
efficient_pre,
efficient_rec,
efficient_f1,
efficient_pred,
efficient_prob
)=evaluate_model(

efficientnet_model,

"EfficientNetB0"

)

Function to plot confusion Matrix

In [ ]:
def plot_confusion_matrix(y_true,
                          y_pred,
                          title):

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    plt.figure(figsize=(7,6))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )

    disp.plot(
        cmap="Blues",
        values_format="d"
    )

    plt.title(title)

    plt.xticks(rotation=45)

    plt.show()

CNN Confusion **Matrix**

In [ ]:
plot_confusion_matrix(

    y_true,

    cnn_pred,

    "CNN Confusion Matrix"

)

**ResNet50**

In [ ]:
plot_confusion_matrix(

    y_true,

    resnet_pred,

    "ResNet50 Confusion Matrix"

)

**MobileNetv2**

In [ ]:
plot_confusion_matrix(

    y_true,

    mobile_pred,

    "MobileNetV2 Confusion Matrix"

)

**EfficientNetB0**

In [ ]:
plot_confusion_matrix(

    y_true,

    efficient_pred,

    "EfficientNetB0 Confusion Matrix"

)

Confusion Matrix **Images**

In [ ]:
models_prediction = {

    "CNN": cnn_pred,

    "ResNet50": resnet_pred,

    "MobileNetV2": mobile_pred,

    "EfficientNetB0": efficient_pred

}

for model_name, prediction in models_prediction.items():

    cm = confusion_matrix(y_true, prediction)

    fig, ax = plt.subplots(figsize=(7,6))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )

    disp.plot(
        cmap="Blues",
        ax=ax,
        values_format="d"
    )

    plt.title(model_name)

    plt.xticks(rotation=45)

    plt.savefig(
        f"{model_name}_ConfusionMatrix.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

Summary **Table**

In [ ]:
summary = pd.DataFrame({

    "Model":[

        "CNN",

        "ResNet50",

        "MobileNetV2",

        "EfficientNetB0"

    ],

    "Accuracy":[

        cnn_acc,

        resnet_acc,

        mobile_acc,

        efficient_acc

    ],

    "Precision":[

        cnn_pre,

        resnet_pre,

        mobile_pre,

        efficient_pre

    ],

    "Recall":[

        cnn_rec,

        resnet_rec,

        mobile_rec,

        efficient_rec

    ],

    "F1 Score":[

        cnn_f1,

        resnet_f1,

        mobile_f1,

        efficient_f1

    ]

})

summary.sort_values(
    by="Accuracy",
    ascending=False
)

***ROC Curve & AUC:***

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

import matplotlib.pyplot as plt
import numpy as np

convert labels to one-hot **format**

In [ ]:
# Number of classes
num_classes = len(class_names)

# Convert labels to one-hot encoding
y_true_bin = label_binarize(
    y_true,
    classes=np.arange(num_classes)
)

print(y_true_bin.shape)

funtion to plot ROC **Curve** **bold text**

In [ ]:
def plot_multiclass_roc(y_true_bin,
                        y_prob,
                        class_names,
                        model_name):

    plt.figure(figsize=(8,6))

    for i in range(len(class_names)):

        fpr, tpr, _ = roc_curve(
            y_true_bin[:, i],
            y_prob[:, i]
        )

        roc_auc = auc(fpr, tpr)

        plt.plot(
            fpr,
            tpr,
            linewidth=2,
            label=f"{class_names[i]} (AUC = {roc_auc:.3f})"
        )

    plt.plot(
        [0,1],
        [0,1],
        linestyle="--"
    )

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")

    plt.title(f"{model_name} ROC Curve")

    plt.legend(loc="lower right")

    plt.grid(True)

    plt.show()

CNN ROC **CURVE**

In [ ]:
plot_multiclass_roc(
    y_true_bin,
    cnn_prob,
    class_names,
    "CNN"
)

RESTNET50 ROC **CURVE**

In [ ]:
plot_multiclass_roc(
    y_true_bin,
    resnet_prob,
    class_names,
    "ResNet50"
)

**MobileNetV2 ROC Curve**

In [ ]:
plot_multiclass_roc(
    y_true_bin,
    mobile_prob,
    class_names,
    "MobileNetV2"
)

EfficientNetB0 ROC Curve

In [ ]:
plot_multiclass_roc(
    y_true_bin,
    efficient_prob,
    class_names,
    "EfficientNetB0"
)

**Calculate Average AUC for Each Model**

In [ ]:
from sklearn.metrics import roc_auc_score

cnn_auc = roc_auc_score(
    y_true_bin,
    cnn_prob,
    multi_class="ovr"
)

resnet_auc = roc_auc_score(
    y_true_bin,
    resnet_prob,
    multi_class="ovr"
)

mobile_auc = roc_auc_score(
    y_true_bin,
    mobile_prob,
    multi_class="ovr"
)

efficient_auc = roc_auc_score(
    y_true_bin,
    efficient_prob,
    multi_class="ovr"
)

**Create AUC Comparison Table**

In [ ]:
auc_df = pd.DataFrame({

    "Model":[
        "CNN",
        "ResNet50",
        "MobileNetV2",
        "EfficientNetB0"
    ],

    "AUC":[
        cnn_auc,
        resnet_auc,
        mobile_auc,
        efficient_auc
    ]

})

auc_df.sort_values(
    by="AUC",
    ascending=False
)

**Plot AUC Comparison**

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(
    auc_df["Model"],
    auc_df["AUC"]
)

plt.title("AUC Comparison")

plt.ylabel("AUC Score")

plt.ylim(0.0, 1.05)

plt.grid(axis="y")

plt.show()

**save AUC results**

In [ ]:
auc_df.to_csv(
    "AUC_Comparison.csv",
    index=False
)

print("AUC comparison saved successfully.")

**Training & Validation Curves**

In [ ]:
def plot_training_history(history, model_name):

    # Accuracy
    plt.figure(figsize=(8,5))

    plt.plot(
        history.history["accuracy"],
        label="Training Accuracy"
    )

    plt.plot(
        history.history["val_accuracy"],
        label="Validation Accuracy"
    )

    plt.title(f"{model_name} Accuracy")

    plt.xlabel("Epoch")

    plt.ylabel("Accuracy")

    plt.legend()

    plt.grid(True)

    plt.show()

    # Loss
    plt.figure(figsize=(8,5))

    plt.plot(
        history.history["loss"],
        label="Training Loss"
    )

    plt.plot(
        history.history["val_loss"],
        label="Validation Loss"
    )

    plt.title(f"{model_name} Loss")

    plt.xlabel("Epoch")

    plt.ylabel("Loss")

    plt.legend()

    plt.grid(True)

    plt.show()

In [ ]:
plot_training_history(cnn_history, "CNN")

plot_training_history(resnet_history, "ResNet50")

plot_training_history(mobilenet_history, "MobileNetV2")

plot_training_history(efficient_history, "EfficientNetB0")

**Grad-CAM**

In [ ]:
import tensorflow as tf

import numpy as np

import cv2

import matplotlib.pyplot as plt

In [ ]:
best_model = tf.keras.models.load_model("EfficientNetB0_Best.keras")

In [ ]:
for layer in best_model.layers:
    print(layer.name)

In [ ]:
last_conv_layer_name = "top_conv"

**Function to Load and Preprocess an Image**

In [ ]:
IMG_SIZE = (224, 224)

def preprocess_image(img_path):

    img = tf.keras.preprocessing.image.load_img(
        img_path,
        target_size=IMG_SIZE
    )

    img_array = tf.keras.preprocessing.image.img_to_array(img)

    img_array = np.expand_dims(img_array, axis=0)

    img_array = img_array / 255.0

    return img_array

**Function to Generate Grad-CAM Heatmap**

In [ ]:
def make_gradcam_heatmap(img_array,
                         model,
                         last_conv_layer_name):

    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[
            model.get_layer(last_conv_layer_name).output,
            model.output
        ]
    )

    with tf.GradientTape() as tape:

        conv_outputs, predictions = grad_model(img_array)

        pred_index = tf.argmax(predictions[0])

        class_channel = predictions[:, pred_index]

    grads = tape.gradient(
        class_channel,
        conv_outputs
    )

    pooled_grads = tf.reduce_mean(
        grads,
        axis=(0,1,2)
    )

    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]

    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap,0)

    heatmap /= tf.math.reduce_max(heatmap)

    return heatmap.numpy()

**Overlay Heatmap on Original Image**

In [ ]:
def overlay_heatmap(img_path,
                    heatmap,
                    alpha=0.4):

    img = cv2.imread(img_path)

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    heatmap = cv2.resize(
        heatmap,
        (img.shape[1], img.shape[0])
    )

    heatmap = np.uint8(255 * heatmap)

    heatmap = cv2.applyColorMap(
        heatmap,
        cv2.COLORMAP_JET
    )

    superimposed = cv2.addWeighted(
        img,
        1-alpha,
        heatmap,
        alpha,
        0
    )

    plt.figure(figsize=(10,4))

    plt.subplot(1,2,1)

    plt.imshow(img)

    plt.title("Original")

    plt.axis("off")

    plt.subplot(1,2,2)

    plt.imshow(superimposed)

    plt.title("Grad-CAM")

    plt.axis("off")

    plt.show()

**Test on a Single Image**

In [ ]:
image_path = "/content/drive/MyDrive/potato_leaf(dataset)/Potato_Leaf_Disease-20260701T072454Z-3-001/Potato___Early_blight/example.jpg"

img_array = preprocess_image(image_path)

heatmap = make_gradcam_heatmap(
    img_array,
    best_model,
    last_conv_layer_name
)

overlay_heatmap(
    image_path,
    heatmap
)

**predict the image**

In [ ]:
prediction = best_model.predict(img_array)

predicted_class = np.argmax(prediction)

confidence = np.max(prediction)

print("Predicted Class :", class_names[predicted_class])

print(f"Confidence : {confidence*100:.2f}%")

**Save the Grad-CAM Image**

In [ ]:
plt.savefig(
    "GradCAM_Result.png",
    dpi=300,
    bbox_inches="tight"
)